# Notebook 3: ViT Face Embeddings + Classifier

**Account:** C | **GPU:** T4 x2 | **Time:** ~8-11 hrs

**Datasets to attach:** `daisee-videos` + `smartlms-openface-features` + `smartlms-scripts`

**Step 1:** Extract ViT-B/16 face crop embeddings from raw videos (~6-8 hrs)
**Step 2:** Train a classifier on those embeddings (~3 hrs)

After this notebook, re-upload the `vit_embeddings/` folder as a new Kaggle dataset
for use in Notebook 5 (Fusion).

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q "numpy<2" transformers xgboost optuna

import torch, os, sys, shutil, glob, subprocess
print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
if torch.cuda.device_count() > 1:
    print(f"Multi-GPU: {torch.cuda.device_count()} GPUs — DataParallel enabled")

In [ ]:
# ── Cell 2: Copy scripts + setup ──
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/app/ml", exist_ok=True)
open(f"{WORK}/app/__init__.py", "w").close()
open(f"{WORK}/app/ml/__init__.py", "w").close()

def find_script(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

scripts = [
    "train_model_v2.py", "train_model_v3.py", "train_model_v4.py",
    "train_multimodal_v5.py", "extract_face_embeddings.py",
    "train_videomae.py", "extract_audio_features.py"
]
for s in scripts:
    found = find_script(s)
    if found:
        shutil.copy(found, f"{WORK}/app/ml/{s}")
        print(f"✓ {s} ← {found}")
    else:
        print(f"✗ NOT FOUND: {s}")

sys.path.insert(0, WORK)
os.chdir(WORK)

In [ ]:
# ── Cell 3: Auto-detect & patch paths ──
print("Attached datasets:")
for d in sorted(os.listdir("/kaggle/input")):
    print(f"  /kaggle/input/{d}/")

# Find video DataSet directory
result = subprocess.run(["find", "/kaggle/input", "-name", "DataSet", "-type", "d"], capture_output=True, text=True)
dataset_dirs = [l for l in result.stdout.strip().split("\n") if l]
VIDEO_DIR = dataset_dirs[0] if dataset_dirs else None
print(f"\nVideo DataSet dir: {VIDEO_DIR}")

# Find Labels
result = subprocess.run(["find", "/kaggle/input", "-name", "AllLabels.csv"], capture_output=True, text=True)
label_hits = [l for l in result.stdout.strip().split("\n") if l]
LABELS_DIR = os.path.dirname(label_hits[0]) if label_hits else None
DAISEE_DIR = os.path.dirname(LABELS_DIR) if LABELS_DIR else None
print(f"Labels dir: {LABELS_DIR}")

# Find OpenFace CSVs
csvs = glob.glob("/kaggle/input/**/openface_output/*.csv", recursive=True)
OPENFACE_DIR = os.path.dirname(csvs[0]) if csvs else None
print(f"OpenFace dir: {OPENFACE_DIR} ({len(csvs)} CSVs)")

# Count videos
if VIDEO_DIR:
    n_avi = len(glob.glob(os.path.join(VIDEO_DIR, "**", "*.avi"), recursive=True))
    print(f"Found {n_avi} .avi video clips")

# Patch extract_face_embeddings.py
embed_path = f"{WORK}/app/ml/extract_face_embeddings.py"
code = open(embed_path).read()
if VIDEO_DIR:
    code = code.replace(
        'DEFAULT_VIDEO_DIR = "/kaggle/input/daisee-videos/DAiSEE/DataSet"',
        f'DEFAULT_VIDEO_DIR = "{VIDEO_DIR}"'
    )
open(embed_path, 'w').write(code)
print("\n✓ Patched extract_face_embeddings.py")

# Patch train_model_v4.py for the ViT classifier step
v4_path = f"{WORK}/app/ml/train_model_v4.py"
code = open(v4_path).read()
if OPENFACE_DIR:
    code = code.replace(
        'OPENFACE_DIR = "/kaggle/input/smartlms-openface/openface_output"',
        f'OPENFACE_DIR = "{OPENFACE_DIR}"'
    )
if DAISEE_DIR:
    code = code.replace(
        'DAISEE_DIR = "/kaggle/input/daisee-dataset"',
        f'DAISEE_DIR = "{DAISEE_DIR}"'
    )
# Point VIT_EMBED_DIR to where we'll save them
code = code.replace(
    'VIT_EMBED_DIR = "/kaggle/input/smartlms-vit-embeddings"',
    'VIT_EMBED_DIR = "/kaggle/working/vit_embeddings"'
)
open(v4_path, 'w').write(code)
print("✓ Patched train_model_v4.py")

# Also patch train_model_v2.py
v2_path = f"{WORK}/app/ml/train_model_v2.py"
if os.path.exists(v2_path) and DAISEE_DIR:
    code = open(v2_path).read()
    code = code.replace(r'C:\Users\revan\Downloads\DAiSEE', DAISEE_DIR)
    open(v2_path, 'w').write(code)
    print("✓ Patched train_model_v2.py")

In [ ]:
# ── Cell 4: Extract ViT-B/16 face embeddings (~6-8 hrs) ──
# Processes all ~8231 video clips, supports resume
import sys
_orig_argv = sys.argv
sys.argv = ['extract_face_embeddings.py', '--resume', '--batch_size', '16', '--sample_fps', '2.0']

from app.ml.extract_face_embeddings import main as embed_main
embed_main()

sys.argv = _orig_argv

import torch, gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ ViT embedding extraction complete!")

In [ ]:
# ── Cell 5: Check embedding extraction ──
import numpy as np

embed_dir = "/kaggle/working/vit_embeddings"
npy_files = glob.glob(os.path.join(embed_dir, "*.npy"))
print(f"Extracted embeddings: {len(npy_files)} clips")

# Quick sanity check
if npy_files:
    sample = np.load(npy_files[0])
    print(f"Sample shape: {sample.shape}")  # Expected: (n_frames, 768)
    print(f"Expected: ~8231 clips")

In [ ]:
# ── Cell 6: Train ViT Embedding Classifier (~3 hrs) ──
# Uses the embeddings we just extracted
from app.ml.train_model_v4 import run_v4_experiment
import torch, gc

run_v4_experiment(mode="vit_train")

gc.collect()
torch.cuda.empty_cache()
print("\n✅ ViT classifier training complete!")

In [ ]:
# ── Cell 7: Review results ──
import json

print("=" * 80)
print("ViT EMBEDDING CLASSIFIER RESULTS")
print("=" * 80)

for rf in sorted(glob.glob("/kaggle/working/experiment_results/*vit*.json")):
    print(f"\n{os.path.basename(rf)}")
    with open(rf) as f:
        data = json.load(f)
    for key, val in data.items():
        if isinstance(val, dict) and 'test_f1_macro' in val:
            print(f"  {key}: F1m = {val['test_f1_macro']:.4f}")
        elif isinstance(val, (int, float)):
            print(f"  {key}: {val}")

In [ ]:
# ── Cell 8: Save outputs ──
shutil.make_archive("/kaggle/working/vit_embeddings_zip", 'zip', "/kaggle/working/vit_embeddings")
shutil.make_archive("/kaggle/working/vit_results", 'zip', "/kaggle/working/experiment_results")

for f in glob.glob("/kaggle/working/*.zip"):
    size_mb = os.path.getsize(f) / 1e6
    print(f"{os.path.basename(f)}: {size_mb:.1f} MB")

print("\n✅ Download from Output tab!")
print("\n⚠ IMPORTANT: Re-upload vit_embeddings/ as a NEW Kaggle dataset")
print("  named 'smartlms-vit-embeddings' for Notebook 5 (Fusion).")